In [ ]:
# ============================================================
# ASSIGNMENT 2 - A2_03 FINALIZE (MIND): NRMS (3 epochs, on HF) + MLP
# Team: Shivam Patel + Suyash Pande | CS4.406 IRE
#
# HF repo donbosoc/mind-artifacts is PUBLIC:
#   - DOWNLOADS use NO token (public read)
#   - UPLOADS use the HF_TOKEN secret (write)
#
# Picks up after the 3-epoch NRMS run: load best checkpoint, final dev eval,
# cache dev scores (for A2_04 hybrid), train the MLP. Uploads guarded so they can't crash.
#
# NEEDS: Kaggle MIND-large mounted. HF_TOKEN secret only needed for the upload steps.
# ============================================================
!pip install huggingface_hub -q
import os, glob, re, numpy as np, pandas as pd, random, pickle
import torch, torch.nn as nn, torch.nn.functional as Fnn
from huggingface_hub import hf_hub_download, HfApi
random.seed(0); torch.manual_seed(0)

device=torch.device("cuda" if torch.cuda.is_available() else "cpu"); print("device:",device)
HF_REPO="donbosoc/mind-artifacts"   # PUBLIC repo

# ---- write token: only needed for uploads; reads work without it ----
WRITE_TOKEN=None
try:
    from kaggle_secrets import UserSecretsClient
    WRITE_TOKEN=UserSecretsClient().get_secret("HF_TOKEN")
    print("write token loaded (uploads enabled)")
except Exception as e:
    print("no HF_TOKEN secret -> downloads still work; uploads will be skipped:", e)

def dl(name):
    # public download: NO token
    return hf_hub_download(HF_REPO, name, repo_type="dataset")

def push(local, name):
    if WRITE_TOKEN is None:
        print("  skip upload (no token):", name); return
    try:
        HfApi(token=WRITE_TOKEN).upload_file(path_or_fileobj=local, path_in_repo=name,
                                             repo_id=HF_REPO, repo_type="dataset")
        print("  pushed", name)
    except Exception as e:
        print("  upload failed for", name, "->", e, "(kept local:", local, ")")

LTR=glob.glob("/kaggle/input/**/MINDlarge_train",recursive=True)[0]
LDV=glob.glob("/kaggle/input/**/MINDlarge_dev",recursive=True)[0]
BEST_CKPT="mind_nrms_best.pt"   # or mind_nrms_ep2.pt for the last epoch specifically
MAX_TITLE=20; MAX_HIST=30


In [ ]:
# ---- rebuild vocab/titles EXACTLY as training (same order -> same ids) ----
NEWS=["news_id","category","subcategory","title","abstract","url","te","ae"]
BEH =["impression_id","user_id","time","history","impressions"]
_WORD=re.compile(r"[^\W\d_]+", re.UNICODE)
def tok(t): return _WORD.findall(t.lower()) if isinstance(t,str) else []
def pfx(x): return f"mind:{x}"
news=pd.concat([pd.read_csv(f"{d}/news.tsv",sep="\t",header=None,names=NEWS,quoting=3,
               usecols=["news_id","title"]) for d in (LTR,LDV)]).drop_duplicates("news_id").reset_index(drop=True)
news["title"]=news["title"].fillna("")
title_tokens={pfx(r.news_id):tok(r.title)[:MAX_TITLE] for r in news.itertuples()}
vocab={"<pad>":0,"<unk>":1}
for tks in title_tokens.values():
    for w in tks:
        if w not in vocab: vocab[w]=len(vocab)
art_ids=[pfx(r.news_id) for r in news.itertuples()]; art_row={a:i for i,a in enumerate(art_ids)}
def enc_title(aid):
    idx=[vocab.get(w,1) for w in title_tokens.get(aid,[])][:MAX_TITLE]
    return idx+[0]*(MAX_TITLE-len(idx))
title_idx=np.array([enc_title(a) for a in art_ids],dtype=np.int64)
print("vocab:",len(vocab),"| titles:",title_idx.shape)


In [ ]:
# ---- NRMS + load best checkpoint (public download, no token) ----
class AdditiveAttention(nn.Module):
    def __init__(s,dim,hid=200):
        super().__init__(); s.p=nn.Linear(dim,hid); s.q=nn.Linear(hid,1,bias=False)
    def forward(s,x,mask=None):
        e=torch.tanh(s.p(x)); a=s.q(e).squeeze(-1)
        if mask is not None: a=a.masked_fill(mask==0,-1e9)
        return (x*torch.softmax(a,-1).unsqueeze(-1)).sum(1)
class NewsEncoder(nn.Module):
    def __init__(s,V,ed=300,h=15,hd=20):
        super().__init__(); d=h*hd
        s.emb=nn.Embedding(V,ed,padding_idx=0); s.attn=nn.MultiheadAttention(ed,h,batch_first=True)
        s.proj=nn.Linear(ed,d); s.add=AdditiveAttention(d); s.drop=nn.Dropout(0.2)
    def forward(s,t):
        m=(t!=0); x=s.drop(s.emb(t)); x,_=s.attn(x,x,x,key_padding_mask=~m)
        x=s.drop(x); x=torch.relu(s.proj(x)); return s.add(x,m)
class UserEncoder(nn.Module):
    def __init__(s,d):
        super().__init__(); s.attn=nn.MultiheadAttention(d,5,batch_first=True); s.add=AdditiveAttention(d)
    def forward(s,hv,mask):
        x,_=s.attn(hv,hv,hv,key_padding_mask=~mask); return s.add(x,mask)
class NRMS(nn.Module):
    def __init__(s,V,h=15,hd=20):
        super().__init__(); s.news=NewsEncoder(V,h=h,hd=hd); s.user=UserEncoder(h*hd)
    def forward(s,ht,hm,ct):
        B,H,L=ht.shape; C=ct.shape[1]
        hv=s.news(ht.reshape(B*H,L)).reshape(B,H,-1); u=s.user(hv,hm)
        cv=s.news(ct.reshape(B*C,L)).reshape(B,C,-1)
        return torch.bmm(cv,u.unsqueeze(-1)).squeeze(-1)

model=NRMS(len(vocab)).to(device)
ck=torch.load(dl(BEST_CKPT), map_location=device)
model.load_state_dict(ck["model"]); print("loaded", BEST_CKPT)


In [ ]:
# ---- metrics + dev eval ----
def load_beh(p):
    b=pd.read_csv(f"{p}/behaviors.tsv",sep="\t",header=None,names=BEH,quoting=3)
    b["t"]=pd.to_datetime(b["time"],format="%m/%d/%Y %I:%M:%S %p",errors="coerce"); return b
b_dv=load_beh(LDV); b_tr=load_beh(LTR)
hist_lut={}
for b in (b_tr,b_dv):
    for u,h in zip(b["user_id"],b["history"]):
        if isinstance(h,str) and h: hist_lut[pfx(u)]=[pfx(x) for x in h.split()]
def hist_tensor(uid):
    h=hist_lut.get(uid,[])[-MAX_HIST:]
    rows=[art_row[a] for a in h if a in art_row][:MAX_HIST]
    mask=[1]*len(rows)+[0]*(MAX_HIST-len(rows)); rows=rows+[0]*(MAX_HIST-len(rows))
    return rows,mask
def _auc(s,lb):
    p=lb==1;n=lb==0;np_,nn=p.sum(),n.sum()
    if np_==0 or nn==0: return None
    o=np.argsort(s);r=np.empty_like(o,float);r[o]=np.arange(1,len(s)+1)
    return float((r[p].sum()-np_*(np_+1)/2)/(np_*nn))
def _mrr(s,lb):
    for rank,idx in enumerate(np.argsort(-s),1):
        if lb[idx]==1: return 1.0/rank
    return 0.0
def _ndcg(s,lb,k):
    o=np.argsort(-s)[:k];g=lb[o]
    dcg=sum(gg/np.log2(i+2) for i,gg in enumerate(g))
    idcg=sum(gg/np.log2(i+2) for i,gg in enumerate(np.sort(lb)[::-1][:k]))
    return float(dcg/idcg) if idcg>0 else 0.0
dev_impr=[]
for iid,u,imps in zip(b_dv["impression_id"],b_dv["user_id"],b_dv["impressions"]):
    if not isinstance(imps,str): continue
    uid=pfx(u)
    if not hist_lut.get(uid): continue
    cand=[pfx(t.split("-")[0]) for t in imps.split()]
    labs=np.array([1 if t.endswith("-1") else 0 for t in imps.split()])
    if labs.sum()>0: dev_impr.append((iid,uid,cand,labs))
random.Random(0).shuffle(dev_impr); dev_impr=dev_impr[:40000]
print("dev eval set:", len(dev_impr))

@torch.no_grad()
def eval_nrms(cache=None):
    model.eval(); A=[];M=[];N5=[];N10=[]
    for iid,uid,cand,labs in dev_impr:
        hr,hm=hist_tensor(uid)
        ht=torch.tensor(title_idx[np.array(hr)]).unsqueeze(0).to(device)
        hmask=torch.tensor(hm,dtype=torch.bool).unsqueeze(0).to(device)
        ct=torch.tensor(title_idx[np.array([art_row.get(c,0) for c in cand])]).unsqueeze(0).to(device)
        s=model(ht,hmask,ct).squeeze(0).cpu().numpy()
        if cache is not None:
            for c,sc in zip(cand,s): cache[(iid,c)]=float(sc)
        a=_auc(s,labs)
        if a is not None: A.append(a)
        M.append(_mrr(s,labs));N5.append(_ndcg(s,labs,5));N10.append(_ndcg(s,labs,10))
    return dict(AUC=np.mean(A),MRR=np.mean(M),nDCG5=np.mean(N5),nDCG10=np.mean(N10))

nrms_cache={}
nrms_metrics=eval_nrms(cache=nrms_cache)
print("=== NRMS (best) dev metrics ===", {k:round(v,4) for k,v in nrms_metrics.items()})
pickle.dump(nrms_cache, open("/kaggle/working/mind_nrms_devscores.pkl","wb"))
push("/kaggle/working/mind_nrms_devscores.pkl","mind_nrms_devscores.pkl")


In [ ]:
# ---- MLP reranker over Q1 features (public npz download) + push ----
d=np.load(dl("mind_large_features.npz"),allow_pickle=True)
Xtr,ytr=d["Xtr"].astype(np.float32),d["ytr"]
Xdv,ydv,gdv=d["Xdv"].astype(np.float32),d["ydv"],[int(g) for g in d["gdv"]]
FEAT=[str(f) for f in d["feat"]]

class MLP(nn.Module):
    def __init__(s,din):
        super().__init__(); s.net=nn.Sequential(nn.Linear(din,128),nn.ReLU(),nn.Dropout(0.2),
                                                 nn.Linear(128,64),nn.ReLU(),nn.Linear(64,1))
    def forward(s,x): return s.net(x).squeeze(-1)
mlp=MLP(Xtr.shape[1]).to(device); opt=torch.optim.Adam(mlp.parameters(),lr=1e-3)
Xt=torch.tensor(Xtr).to(device); yt=torch.tensor(ytr.astype(np.float32)).to(device); bs=100000
for ep in range(3):
    mlp.train(); perm=torch.randperm(len(Xt))
    for i in range(0,len(Xt),bs):
        idx=perm[i:i+bs]; loss=Fnn.binary_cross_entropy_with_logits(mlp(Xt[idx]),yt[idx])
        opt.zero_grad(); loss.backward(); opt.step()
    print(f"MLP epoch {ep} loss {loss.item():.4f}")
mlp.eval()
with torch.no_grad():
    mlp_scores=mlp(torch.tensor(Xdv).to(device)).cpu().numpy()
def eval_flat(scores,y,g):
    A=[];M=[];N5=[];N10=[];pos=0
    for gi in g:
        s=scores[pos:pos+gi];lb=y[pos:pos+gi];pos+=gi
        a=_auc(s,lb)
        if a is not None:A.append(a)
        M.append(_mrr(s,lb));N5.append(_ndcg(s,lb,5));N10.append(_ndcg(s,lb,10))
    return dict(AUC=np.mean(A),MRR=np.mean(M),nDCG5=np.mean(N5),nDCG10=np.mean(N10))
mlp_metrics=eval_flat(mlp_scores,ydv,gdv)
print("=== MLP dev metrics ===", {k:round(v,4) for k,v in mlp_metrics.items()})
torch.save(mlp.state_dict(),"/kaggle/working/mind_mlp.pt")
push("/kaggle/working/mind_mlp.pt","mind_mlp.pt")

print("\n=== NEURAL RERANKERS SUMMARY (MIND dev) ===")
print("NRMS (3 epochs):", {k:round(v,4) for k,v in nrms_metrics.items()})
print("MLP           :", {k:round(v,4) for k,v in mlp_metrics.items()})
